# LeetCode 3624 — Number of Integers With Popcount-Depth Equal to K II bằng Python

- Link: https://leetcode.com/problems/number-of-integers-with-popcount-depth-equal-to-k-ii/
- Độ khó: **Hard**
- Chủ đề: **Fenwick Tree** (nhiều cây theo "loại"), cập nhật điểm + đếm đoạn, chia căn, Segment Tree lưu vectơ đếm

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def popcountDepth(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra mỗi phần tử chỉ cần **một con số nhỏ**: độ sâu popcount `d ∈ [0, 5]` (vì `nums[i] ≤ 10^{15}`).
2. Biến bài thành: mảng nhãn `d[i]`, truy vấn "đếm số `i ∈ [l, r]` có `d[i] = k`", cập nhật "đổi nhãn tại `i`".
3. Dùng **6 Fenwick Tree** (mỗi nhãn một cây): truy vấn và cập nhật đều `O(log n)`.
4. So sánh với chia căn (đếm theo khối) và Segment Tree lưu vectơ 6 phần tử.
5. Kiểm chứng bằng mô phỏng trực tiếp.

## 2. Đề bài & ví dụ

Độ sâu popcount của `x`: số bước `x → popcount(x) → …` để về 1. Truy vấn:

- `[1, l, r, k]`: đếm `j ∈ [l, r]` có độ sâu của `nums[j]` bằng `k`.
- `[2, idx, val]`: gán `nums[idx] = val`.

**Ví dụ 1:** `nums = [2,4]`, `queries = [[1,0,1,1],[2,1,1],[1,0,1,0]]` → `[2, 1]`.

**Ví dụ 2:** `nums = [3,5,6]`, `queries = [[1,0,2,2],[2,1,4],[1,1,2,1],[1,0,1,0]]` → `[3, 1, 0]`.

**Ví dụ 3:** `nums = [1,2]`, `queries = [[1,0,1,1],[2,0,3],[1,0,0,1],[1,0,0,2]]` → `[1, 0, 1]`.

**Ràng buộc:** `n, q ≤ 10^5`, `nums[i], val ≤ 10^{15}`, `0 ≤ k ≤ 5`.

In [1]:
from typing import List
import random, time

examples = [
    ([2, 4], [[1, 0, 1, 1], [2, 1, 1], [1, 0, 1, 0]], [2, 1]),
    ([3, 5, 6], [[1, 0, 2, 2], [2, 1, 4], [1, 1, 2, 1], [1, 0, 1, 0]], [3, 1, 0]),
    ([1, 2], [[1, 0, 1, 1], [2, 0, 3], [1, 0, 0, 1], [1, 0, 0, 2]], [1, 0, 1]),
]
for a, qs, exp in examples:
    print(a, qs, "-> kỳ vọng", exp)

[2, 4] [[1, 0, 1, 1], [2, 1, 1], [1, 0, 1, 0]] -> kỳ vọng [2, 1]
[3, 5, 6] [[1, 0, 2, 2], [2, 1, 4], [1, 1, 2, 1], [1, 0, 1, 0]] -> kỳ vọng [3, 1, 0]
[1, 2] [[1, 0, 1, 1], [2, 0, 3], [1, 0, 0, 1], [1, 0, 0, 2]] -> kỳ vọng [1, 0, 1]


## 3. Kiến thức nền

### 3.1. Độ sâu popcount và giá trị lớn nhất

`x ≤ 10^{15} < 2^{50}` ⇒ `popcount(x) ≤ 50` ⇒ độ sâu của `popcount(x)` `≤ 3` ⇒ độ sâu của `x` `≤ 4`. Tính độ sâu bằng vòng lặp: tối đa 5 lần `bit_count`.

In [2]:
def depth(x):
    d = 0
    while x != 1:
        x = bin(x).count("1"); d += 1
    return d

print([depth(x) for x in [1, 2, 3, 7, 127, 2**50 - 1, 10**15]])

[0, 1, 2, 3, 4, 4, 3]


### 3.2. Fenwick Tree đếm

`add(i, ±1)` và `prefix(i)` trong `O(log n)`; đếm đoạn `[l, r]` = `prefix(r+1) − prefix(l)`.

### 3.3. Nhiều cây theo nhãn

Có 6 nhãn ⇒ 6 cây Fenwick, mỗi cây đánh dấu vị trí có nhãn tương ứng. Bộ nhớ `6 · (n + 1)` số — nhỏ.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giá trị `nums[i]` lớn nhưng **thông tin cần** chỉ là nhãn `d[i]`. Mỗi cập nhật đổi đúng một nhãn ⇒ trừ 1 ở cây nhãn cũ, cộng 1 ở cây nhãn mới. Mỗi truy vấn đọc tổng đoạn trên cây nhãn `k`.

Ví dụ 2: nhãn ban đầu `[2, 2, 2]` (`3, 5, 6` đều có 2 bit 1 ⇒ `→ 2 → 1`). Truy vấn `[0,2,2]` ⇒ 3. Gán `nums[1] = 4` ⇒ nhãn `[2, 1, 2]`. Truy vấn `[1,2,1]` ⇒ 1; `[0,1,0]` ⇒ 0. ✔

## 5. Cách 1 — Brute Force: mô phỏng

In [3]:
def brute(nums, queries):
    a = nums[:]; out = []
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q; out.append(sum(1 for j in range(l, r + 1) if depth(a[j]) == k))
        else:
            a[q[1]] = q[2]
    return out

for a, qs, exp in examples:
    print(brute(a, qs), "kỳ vọng", exp)

[2, 1] kỳ vọng [2, 1]
[3, 1, 0] kỳ vọng [3, 1, 0]
[1, 0, 1] kỳ vọng [1, 0, 1]


**Nút thắt:** `O(n)` mỗi truy vấn (và tính lại độ sâu mỗi lần) ⇒ `10^{10}`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `d[i] = depth(nums[i])` một lần.
2. Dựng 6 Fenwick Tree (dựng `O(n)` bằng cách cộng dồn lên cha).
3. Truy vấn loại 1: nếu `k > 5` trả 0, ngược lại tổng đoạn trên cây `k`.
4. Truy vấn loại 2: `nd = depth(val)`; nếu khác `d[idx]` thì cập nhật hai cây.

Tổng `O((n + q) log n)`.

## 7. Các cách tiếp theo

- **Cách 2:** chia căn — mỗi khối lưu mảng đếm 6 nhãn; truy vấn `O(√n)`, cập nhật `O(1)`.
- **Cách 3:** 6 Fenwick Tree (lời giải nộp).
- **Cách 4:** Segment Tree mỗi nút lưu **vectơ đếm** 6 phần tử — tổng quát hoá được cho nhiều thao tác hơn (ví dụ gán đoạn).

### 7.1. Cách 2 — Chia căn

In [4]:
def sqrt_decomp(nums, queries):
    n = len(nums); B = max(1, int(n ** 0.5)); d = [depth(x) for x in nums]
    blk = [[0] * 6 for _ in range((n + B - 1) // B)]
    for i, v in enumerate(d): blk[i // B][v] += 1
    out = []
    for q in queries:
        if q[0] == 2:
            _, i, v = q; nd = depth(v)
            blk[i // B][d[i]] -= 1; blk[i // B][nd] += 1; d[i] = nd
        else:
            _, l, r, k = q; res = 0
            if k > 5: out.append(0); continue
            while l <= r and l % B: res += d[l] == k; l += 1
            while l + B - 1 <= r: res += blk[l // B][k]; l += B
            while l <= r: res += d[l] == k; l += 1
            out.append(res)
    return out

for a, qs, exp in examples:
    print(sqrt_decomp(a, qs), "kỳ vọng", exp)

[2, 1] kỳ vọng [2, 1]
[3, 1, 0] kỳ vọng [3, 1, 0]
[1, 0, 1] kỳ vọng [1, 0, 1]


### 7.2. Cách 3 — 6 Fenwick Tree (lời giải chính)

In [5]:
def fenwicks(nums, queries):
    n = len(nums); d = [depth(x) for x in nums]
    T = [[0] * (n + 1) for _ in range(6)]
    for i, v in enumerate(d): T[v][i + 1] += 1
    for t in T:                                               # dựng O(n)
        for i in range(1, n + 1):
            j = i + (i & -i)
            if j <= n: t[j] += t[i]
    def add(t, i, v):
        i += 1
        while i <= n: t[i] += v; i += i & -i
    def pre(t, i):
        s = 0
        while i > 0: s += t[i]; i -= i & -i
        return s
    out = []
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q
            out.append(pre(T[k], r + 1) - pre(T[k], l) if k <= 5 else 0)
        else:
            _, i, v = q; nd = depth(v)
            if nd != d[i]: add(T[d[i]], i, -1); add(T[nd], i, 1); d[i] = nd
    return out

for a, qs, exp in examples:
    print(fenwicks(a, qs), "kỳ vọng", exp)

[2, 1] kỳ vọng [2, 1]
[3, 1, 0] kỳ vọng [3, 1, 0]
[1, 0, 1] kỳ vọng [1, 0, 1]


### 7.3. Cách 4 — Segment Tree lưu vectơ đếm

In [6]:
def segtree_vectors(nums, queries):
    n = len(nums); size = 1
    while size < n: size *= 2
    t = [[0] * 6 for _ in range(2 * size)]
    for i, x in enumerate(nums): t[size + i][depth(x)] = 1
    for i in range(size - 1, 0, -1): t[i] = [a + b for a, b in zip(t[2 * i], t[2 * i + 1])]
    out = []
    for q in queries:
        if q[0] == 2:
            _, i, v = q; p = size + i; t[p] = [0] * 6; t[p][depth(v)] = 1; p //= 2
            while p: t[p] = [a + b for a, b in zip(t[2 * p], t[2 * p + 1])]; p //= 2
        else:
            _, l, r, k = q
            if k > 5: out.append(0); continue
            res = 0; l += size; r += size + 1
            while l < r:
                if l & 1: res += t[l][k]; l += 1
                if r & 1: r -= 1; res += t[r][k]
                l //= 2; r //= 2
            out.append(res)
    return out

for a, qs, exp in examples:
    print(segtree_vectors(a, qs), "kỳ vọng", exp)

[2, 1] kỳ vọng [2, 1]
[3, 1, 0] kỳ vọng [3, 1, 0]
[1, 0, 1] kỳ vọng [1, 0, 1]


## 8. TRACE từng bước

In [7]:
def trace(nums, queries):
    d = [depth(x) for x in nums]
    print("nums =", nums, "| nhãn độ sâu =", d)
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q
            hits = [j for j in range(l, r + 1) if d[j] == k]
            print(f"  hỏi [{l},{r}] độ sâu {k}: vị trí {hits} -> {len(hits)}")
        else:
            _, i, v = q; old = d[i]; d[i] = depth(v); nums = nums[:i] + [v] + nums[i + 1:]
            print(f"  gán nums[{i}] = {v} ({bin(v)[2:]}): nhãn {old} -> {d[i]}; cây {old} trừ 1, cây {d[i]} cộng 1")

trace([3, 5, 6], [[1, 0, 2, 2], [2, 1, 4], [1, 1, 2, 1], [1, 0, 1, 0]])

nums = [3, 5, 6] | nhãn độ sâu = [2, 2, 2]
  hỏi [0,2] độ sâu 2: vị trí [0, 1, 2] -> 3
  gán nums[1] = 4 (100): nhãn 2 -> 1; cây 2 trừ 1, cây 1 cộng 1
  hỏi [1,2] độ sâu 1: vị trí [1] -> 1
  hỏi [0,1] độ sâu 0: vị trí [] -> 0


In [8]:
trace([1, 7, 127, 8, 255], [[1, 0, 4, 3], [1, 0, 4, 4], [2, 3, 2**49 - 1], [1, 0, 4, 3]])

nums = [1, 7, 127, 8, 255] | nhãn độ sâu = [0, 3, 4, 1, 2]
  hỏi [0,4] độ sâu 3: vị trí [1] -> 1
  hỏi [0,4] độ sâu 4: vị trí [2] -> 1
  gán nums[3] = 562949953421311 (1111111111111111111111111111111111111111111111111): nhãn 1 -> 4; cây 1 trừ 1, cây 4 cộng 1
  hỏi [0,4] độ sâu 3: vị trí [1] -> 1


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def popcountDepth(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        def depth(x):
            d = 0
            while x != 1:
                x = bin(x).count('1')
                d += 1
            return d

        n = len(nums)
        d = [depth(x) for x in nums]
        T = [[0] * (n + 1) for _ in range(6)]
        for i, v in enumerate(d):
            T[v][i + 1] += 1
        for t in T:
            for i in range(1, n + 1):
                j = i + (i & -i)
                if j <= n:
                    t[j] += t[i]
        out = []
        for q in queries:
            if q[0] == 1:
                _, l, r, k = q
                if k > 5:
                    out.append(0)
                    continue
                t = T[k]
                s = 0
                i = r + 1
                while i > 0:
                    s += t[i]; i -= i & -i
                i = l
                while i > 0:
                    s -= t[i]; i -= i & -i
                out.append(s)
            else:
                _, idx, val = q
                nd = depth(val)
                if nd != d[idx]:
                    t = T[d[idx]]
                    i = idx + 1
                    while i <= n:
                        t[i] -= 1; i += i & -i
                    t = T[nd]
                    i = idx + 1
                    while i <= n:
                        t[i] += 1; i += i & -i
                    d[idx] = nd
        return out

sol = Solution()
for a, qs, exp in examples:
    got = sol.popcountDepth(a, qs)
    print(got, "✔" if got == exp else "✘")

[2, 1] ✔
[3, 1, 0] ✔
[1, 0, 1] ✔


## 10. Vì sao thuật toán đúng

1. **Bất biến:** sau mỗi truy vấn, với mọi nhãn `c`, cây `T[c]` là Fenwick của mảng chỉ thị `[d[i] = c]`, và `d[i] = depth(nums[i])` hiện tại.
2. **Khởi tạo** đúng theo cách dựng (mỗi nút `i` cộng dồn vào nút cha `i + lowbit(i)`).
3. **Cập nhật** `[2, idx, val]`: chỉ thị của vị trí `idx` đổi từ nhãn cũ sang nhãn mới ⇒ `−1` / `+1` đúng hai cây (bỏ qua nếu nhãn không đổi).
4. **Truy vấn** `[1, l, r, k]`: tổng chỉ thị trên `[l, r]` của cây `k` = số vị trí có nhãn `k`. Nhãn luôn `≤ 4 < 6` ⇒ `k = 5` cho 0 (cây rỗng) — vẫn đúng; kiểm tra `k > 5` chỉ là phòng hờ.

## 11. Phân tích Time / Space Complexity

| Cách | Khởi tạo | Truy vấn | Cập nhật | Space |
|---|---|---|---|---|
| 1. Mô phỏng | — | `O(n · 5)` | `O(1)` | `O(n)` |
| 2. Chia căn | `O(n)` | `O(√n)` | `O(1)` | `O(n)` |
| 3. 6 Fenwick | `O(6n)` | `O(log n)` | `O(log n)` | `O(6n)` |
| 4. Segment Tree vectơ | `O(6n)` | `O(log n)` | `O(6 log n)` | `O(12n)` |

## 12. Bộ test

In [10]:
METHODS = [brute, sqrt_decomp, fenwicks, segtree_vectors, lambda a, q: Solution().popcountDepth(a, q)]
for a, qs, exp in examples:
    assert all(f(a, qs) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().popcountDepth([1], [[1, 0, 0, 0], [1, 0, 0, 5]]) == [1, 0]
assert Solution().popcountDepth([10**15], [[1, 0, 0, depth(10**15)], [2, 0, 1], [1, 0, 0, 0]]) == [1, 1]
assert Solution().popcountDepth([127, 127], [[1, 0, 1, 4], [2, 0, 127], [1, 0, 1, 4]]) == [2, 2]
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3624)
pool = [1, 2, 3, 5, 7, 11, 127, 255, 2**31 - 1, 10**15, 999, 2**49 - 1]
for _ in range(500):
    n = rng.randint(1, 15); a = [rng.choice(pool) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 20)):
        if rng.random() < 0.4: qs.append([2, rng.randrange(n), rng.choice(pool + [rng.randint(1, 10**15)])])
        else:
            l = rng.randrange(n); qs.append([1, l, rng.randrange(l, n), rng.randint(0, 5)])
    b = brute(a, qs)
    assert all(f(a, qs) == b for f in METHODS[1:]), (a, qs)
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
n = q = 10**5
a = [rng.randint(1, 10**15) for _ in range(n)]; qs = []
for _ in range(q):
    if rng.random() < 0.5: qs.append([2, rng.randrange(n), rng.randint(1, 10**15)])
    else:
        l = rng.randrange(n); qs.append([1, l, rng.randrange(l, n), rng.randint(0, 5)])
t0 = time.time(); x = Solution().popcountDepth(a, qs); t1 = time.time()
y = sqrt_decomp(a, qs); t2 = time.time()
print(f"Fenwick: {t1 - t0:.2f}s | chia căn: {t2 - t1:.2f}s | khớp: {x == y}")

Fenwick: 0.32s | chia căn: 1.39s | khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cập nhật cây mới nhưng quên trừ ở cây cũ

In [14]:
def wrong_no_remove(nums, queries):
    n = len(nums); d = [depth(x) for x in nums]; cnt = [[0] * n for _ in range(6)]
    for i, v in enumerate(d): cnt[v][i] = 1
    out = []
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q; out.append(sum(cnt[k][l:r + 1]))
        else:
            _, i, v = q; nd = depth(v); cnt[nd][i] = 1; d[i] = nd                    # SAI: không xoá nhãn cũ
    return out

a, qs = [3], [[2, 0, 4], [1, 0, 0, 2]]
print("sai:", wrong_no_remove(a, qs), "| đúng:", brute(a, qs))
assert wrong_no_remove(a, qs) != brute(a, qs)

sai: [1] | đúng: [0]


### Lỗi 2 — Độ sâu của `1` tính là 1 (vòng lặp do-while)

In [15]:
def wrong_depth(x):
    d = 0
    while True:
        x = bin(x).count("1"); d += 1
        if x == 1: return d                                                         # SAI: depth(1) = 1

print("sai:", wrong_depth(1), "| đúng:", depth(1))

sai: 1 | đúng: 0


### Lỗi 3 — Chỉ số Fenwick lệch (dùng `prefix(r) − prefix(l − 1)` với cây 1-indexed nhưng không cộng 1)

In [16]:
def wrong_index(nums, queries):
    n = len(nums); d = [depth(x) for x in nums]; T = [[0] * (n + 1) for _ in range(6)]
    def add(t, i, v):
        while i <= n and i > 0: t[i] += v; i += i & -i                              # SAI: không +1, i = 0 bị bỏ
    def pre(t, i):
        s = 0
        while i > 0: s += t[i]; i -= i & -i
        return s
    for i, v in enumerate(d): add(T[v], i, 1)
    out = []
    for q in queries:
        if q[0] == 1: _, l, r, k = q; out.append(pre(T[k], r) - pre(T[k], l - 1) if l > 0 else pre(T[k], r))
        else: _, i, v = q; add(T[d[i]], i, -1); d[i] = depth(v); add(T[d[i]], i, 1)
    return out

a, qs, exp = examples[0]
print("sai:", wrong_index(a, qs), "| đúng:", exp)

sai: [1, 1] | đúng: [2, 1]


### Lỗi 4 — Gán xong cập nhật cây nhưng **không cập nhật** `d[idx]`

Lần gán tiếp theo ở cùng vị trí sẽ trừ nhầm cây.

In [17]:
def wrong_stale_label(nums, queries):
    n = len(nums); d = [depth(x) for x in nums]; cnt = [[0] * n for _ in range(6)]
    for i, v in enumerate(d): cnt[v][i] = 1
    out = []
    for q in queries:
        if q[0] == 1: _, l, r, k = q; out.append(sum(cnt[k][l:r + 1]))
        else: _, i, v = q; cnt[d[i]][i] = 0; cnt[depth(v)][i] = 1                    # SAI: quên d[i] = depth(v)
    return out

qs = [[2, 0, 3], [2, 0, 1], [1, 0, 0, 2]]
print("sai:", wrong_stale_label([2], qs), "| đúng:", brute([2], qs))

sai: [1] | đúng: [0]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Giá trị lớn nhưng truy vấn chỉ quan tâm một **thuộc tính nhỏ** ⇒ thay giá trị bằng nhãn.
- "Đếm phần tử có nhãn `k` trong đoạn" + cập nhật điểm + ít nhãn ⇒ **một Fenwick mỗi nhãn**.
- Nhiều nhãn (lớn) ⇒ chia căn với bảng đếm theo khối, hoặc xử lý offline theo nhãn.
- Dựng Fenwick `O(n)` bằng cách cộng dồn lên cha — nhanh hơn `n` lần `add`.

## 15. Biến thể của bài

1. **Bản I**: đếm số `x ∈ [1, n]` có độ sâu `k` (tổ hợp theo bit).
2. **Gán cả đoạn** `[l, r] = val`: Segment Tree vectơ + lazy "gán nhãn".
3. **Truy vấn nhãn xuất hiện nhiều nhất trong đoạn**: Segment Tree vectơ, lấy argmax của tổng.
4. **Nhãn = số bit 1** (tới 50 nhãn): 51 cây Fenwick vẫn ổn, hoặc chia căn.

In [18]:
# Biến thể 3: nhãn độ sâu xuất hiện nhiều nhất trong đoạn (hoà -> nhãn nhỏ nhất)
def most_common_depth(nums, queries):
    n = len(nums); size = 1
    while size < n: size *= 2
    t = [[0] * 6 for _ in range(2 * size)]
    for i, x in enumerate(nums): t[size + i][depth(x)] = 1
    for i in range(size - 1, 0, -1): t[i] = [a + b for a, b in zip(t[2 * i], t[2 * i + 1])]
    out = []
    for l, r in queries:
        acc = [0] * 6; l += size; r += size + 1
        while l < r:
            if l & 1: acc = [a + b for a, b in zip(acc, t[l])]; l += 1
            if r & 1: r -= 1; acc = [a + b for a, b in zip(acc, t[r])]
            l //= 2; r //= 2
        out.append(max(range(6), key=lambda c: (acc[c], -c)))
    return out

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 12); a = [rng.choice([1, 2, 3, 7, 127, 255]) for _ in range(n)]; qs = []
    for _ in range(5):
        l = rng.randrange(n); qs.append((l, rng.randrange(l, n)))
    exp = []
    for l, r in qs:
        c = [0] * 6
        for j in range(l, r + 1): c[depth(a[j])] += 1
        exp.append(max(range(6), key=lambda x: (c[x], -x)))
    assert most_common_depth(a, qs) == exp
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Truy vấn | Khi nào dùng |
|---|---|---|---|
| 1. Mô phỏng | Đếm lại | `O(n)` | Kiểm chứng |
| 2. Chia căn | Bảng đếm theo khối | `O(√n)` | Nhiều nhãn |
| 3. 6 Fenwick | Một cây mỗi nhãn | `O(log n)` | **Lời giải chuẩn** |
| 4. Segment Tree vectơ | Tổng vectơ | `O(log n)` | Thao tác đoạn phức tạp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [1, 1, 1]`, `queries = [[1,0,2,0],[2,1,2],[1,0,2,0]]`.

**Bài 2 (tính tay).** `nums = [7, 8, 9]`, `queries = [[1,0,2,3],[1,0,2,2],[1,0,2,1]]`.

**Bài 3 (điền TODO).** Viết `count_at_least(nums, queries)`: truy vấn `[1, l, r, k]` đếm số phần tử có độ sâu **≥ k**.

**Bài 4 (tự viết lại).** Viết lớp `DepthArray` với `update(i, v)` và `count(l, r, k)` dùng Segment Tree vectơ.

**Bài 5 (mini challenge).** Thêm truy vấn `[3, l, r, val]` gán cả đoạn — cài Segment Tree vectơ có lazy.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_at_least(nums, queries):
    # TODO: cộng các cây k..5
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def with_range_assign(nums, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `depth(1) = 0` ⇒ 3. (2) Gán `nums[1] = 2` (độ sâu 1). (3) `[3, 2]`.

**Bài 2.** (1) `7 → 3 → 2 → 1`: 3; `8 → 1`: 1; `9 = 1001 → 2 → 1`: 2. (2) Nhãn `[3, 1, 2]`. (3) `[1, 1, 1]`.

**Bài 3.** (1) `Σ_{c ≥ k}` của các cây. (2) Hoặc thêm cây "≥ k". (3) So mô phỏng.

**Bài 4.** (1) Dùng lại Cách 4. (2) Đóng gói thành lớp. (3) Kiểm chứng ngẫu nhiên.

**Bài 5.** (1) Lazy là "gán nhãn `c` cho cả nút". (2) Khi đẩy xuống: vectơ con = `[0..0, độ dài, 0..0]`. (3) Truy vấn như cũ, đẩy lazy trước khi đi xuống.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().popcountDepth([1, 1, 1], [[1, 0, 2, 0], [2, 1, 2], [1, 0, 2, 0]]) == [3, 2]
assert Solution().popcountDepth([7, 8, 9], [[1, 0, 2, 3], [1, 0, 2, 2], [1, 0, 2, 1]]) == [1, 1, 1]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def count_at_least_sol(nums, queries):
    res = []; a = nums[:]
    sub = []
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q
            res.append(sum(fenwicks(a, [[1, l, r, c]])[0] for c in range(k, 6)))
        else:
            a[q[1]] = q[2]
    return res

class DepthArray:
    def __init__(self, nums):
        self.n = len(nums); self.size = 1
        while self.size < self.n: self.size *= 2
        self.t = [[0] * 6 for _ in range(2 * self.size)]
        for i, x in enumerate(nums): self.t[self.size + i][depth(x)] = 1
        for i in range(self.size - 1, 0, -1): self.t[i] = [a + b for a, b in zip(self.t[2 * i], self.t[2 * i + 1])]
    def update(self, i, v):
        p = self.size + i; self.t[p] = [0] * 6; self.t[p][depth(v)] = 1; p //= 2
        while p: self.t[p] = [a + b for a, b in zip(self.t[2 * p], self.t[2 * p + 1])]; p //= 2
    def count(self, l, r, k):
        res = 0; l += self.size; r += self.size + 1
        while l < r:
            if l & 1: res += self.t[l][k]; l += 1
            if r & 1: r -= 1; res += self.t[r][k]
            l //= 2; r //= 2
        return res

rng = random.Random(4)
for _ in range(100):
    n = rng.randint(1, 10); a = [rng.choice([1, 2, 3, 7, 127]) for _ in range(n)]; qs = []; da = DepthArray(a); b = a[:]
    for _ in range(10):
        if rng.random() < 0.4:
            i = rng.randrange(n); v = rng.choice([1, 2, 3, 7, 127]); qs.append([2, i, v]); da.update(i, v); b[i] = v
        else:
            l = rng.randrange(n); r = rng.randrange(l, n); k = rng.randint(0, 5); qs.append([1, l, r, k])
            assert da.count(l, r, k) == sum(1 for j in range(l, r + 1) if depth(b[j]) == k)
    exp = []; c = a[:]
    for q in qs:
        if q[0] == 1: exp.append(sum(1 for j in range(q[1], q[2] + 1) if depth(c[j]) >= q[3]))
        else: c[q[1]] = q[2]
    assert count_at_least_sol(a, qs) == exp
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: gán đoạn bằng Segment Tree vectơ có lazy
def with_range_assign_sol(nums, queries):
    n = len(nums); t = [[0] * 6 for _ in range(4 * n)]; lz = [-1] * (4 * n)
    def build(x, lo, hi):
        if lo == hi: t[x] = [0] * 6; t[x][depth(nums[lo])] = 1; return
        m = (lo + hi) // 2; build(2 * x, lo, m); build(2 * x + 1, m + 1, hi); t[x] = [a + b for a, b in zip(t[2 * x], t[2 * x + 1])]
    def apply(x, lo, hi, c):
        t[x] = [0] * 6; t[x][c] = hi - lo + 1; lz[x] = c
    def push(x, lo, hi):
        if lz[x] != -1:
            m = (lo + hi) // 2; apply(2 * x, lo, m, lz[x]); apply(2 * x + 1, m + 1, hi, lz[x]); lz[x] = -1
    def assign(x, lo, hi, l, r, c):
        if r < lo or hi < l: return
        if l <= lo and hi <= r: apply(x, lo, hi, c); return
        push(x, lo, hi); m = (lo + hi) // 2
        assign(2 * x, lo, m, l, r, c); assign(2 * x + 1, m + 1, hi, l, r, c)
        t[x] = [a + b for a, b in zip(t[2 * x], t[2 * x + 1])]
    def query(x, lo, hi, l, r, k):
        if r < lo or hi < l: return 0
        if l <= lo and hi <= r: return t[x][k]
        push(x, lo, hi); m = (lo + hi) // 2
        return query(2 * x, lo, m, l, r, k) + query(2 * x + 1, m + 1, hi, l, r, k)
    build(1, 0, n - 1); out = []
    for q in queries:
        if q[0] == 1: out.append(query(1, 0, n - 1, q[1], q[2], q[3]))
        elif q[0] == 2: assign(1, 0, n - 1, q[1], q[1], depth(q[2]))
        else: assign(1, 0, n - 1, q[1], q[2], depth(q[3]))
    return out

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 10); a = [rng.choice([1, 2, 3, 7, 127]) for _ in range(n)]; qs = []
    for _ in range(12):
        r_ = rng.random(); l = rng.randrange(n); r = rng.randrange(l, n)
        if r_ < 0.3: qs.append([3, l, r, rng.choice([1, 2, 3, 7, 127])])
        elif r_ < 0.5: qs.append([2, l, rng.choice([1, 2, 3, 7])])
        else: qs.append([1, l, r, rng.randint(0, 4)])
    b = a[:]; exp = []
    for q in qs:
        if q[0] == 1: exp.append(sum(1 for j in range(q[1], q[2] + 1) if depth(b[j]) == q[3]))
        elif q[0] == 2: b[q[1]] = q[2]
        else:
            for j in range(q[1], q[2] + 1): b[j] = q[3]
    assert with_range_assign_sol(a, qs) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
d[i] = depth(nums[i])                      # <= 5 bước popcount
T[c] = Fenwick chỉ thị [d[i] == c], c = 0..5
[1, l, r, k] -> T[k].sum(l, r)  (k > 5 -> 0)
[2, i, v]    -> nd = depth(v); nếu khác: T[d[i]].add(i, -1); T[nd].add(i, +1); d[i] = nd
```

**Checklist:**

- [ ] `depth(1) = 0`?
- [ ] Trừ ở cây cũ **và** cập nhật `d[i]`?
- [ ] Chỉ số Fenwick 1-indexed đúng?
- [ ] `k` ngoài phạm vi trả 0?
- [ ] Đã so với mô phỏng?